In [12]:
device

device(type='mps')

In [11]:
import torch
import numpy as np
from torchgeo.models import tessera, Tessera_Weights

# ─────────────────────────────────────────────
# 1.  LOAD THE PRETRAINED TESSERA MODEL
# ─────────────────────────────────────────────
# The combined encoder outputs a 128-dim embedding (fusion MLP output,
# before the projector). Weights are CC0-licensed.

model = tessera(weights=Tessera_Weights.TESSERA)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "mps")
model = model.to(device)

# ─────────────────────────────────────────────
# 2.  YOUR INPUT DATA (adjust to your format)
# ─────────────────────────────────────────────
# Assumptions for a single pixel:
#   s2_values : np.ndarray  shape (T_s2, 10)  — 10 S2 bands per timestep
#   s1_values : np.ndarray  shape (T_s1, 2)   — [VV, VH] per timestep
#   s2_doys   : np.ndarray  shape (T_s2,)     — day-of-year for each S2 obs
#   s1_doys   : np.ndarray  shape (T_s1,)     — day-of-year for each S1 obs
#
# Replace with your actual arrays:
S2_BANDS = ['B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12']

s2_values = ...  # (T_s2, 10)  — reflectance, already cloud-filtered
s1_values = ...  # (T_s1, 2)   — [VV, VH] backscatter in linear or dB scale
s2_doys   = ...  # (T_s2,)     — int, 1–365
s1_doys   = ...  # (T_s1,)     — int, 1–365

# ─────────────────────────────────────────────
# 3.  HELPER: BUILD A D-PIXEL TENSOR
#     Each timestep = [spectral bands] + [sin(DOY), cos(DOY)]
#     Exactly 40 timesteps, sampled with replacement if needed.
# ─────────────────────────────────────────────
N_TIMESTEPS = 40   # fixed sequence length used at training

def build_dpixel(values: np.ndarray, doys: np.ndarray,
                 n_steps: int = N_TIMESTEPS) -> torch.Tensor:
    """
    Sample n_steps timesteps (with replacement if T < n_steps),
    append sin/cos DOY positional encoding, return tensor (1, T, C+2).
    """
    T = len(doys)
    idx = np.random.choice(T, size=n_steps, replace=(T < n_steps))
    sampled_vals = values[idx]          # (40, C)
    sampled_doys = doys[idx]            # (40,)

    # Normalise DOY to [0, 2π] and encode
    doy_rad = 2 * np.pi * sampled_doys / 365.0
    doy_sin = np.sin(doy_rad)[:, None]  # (40, 1)
    doy_cos = np.cos(doy_rad)[:, None]  # (40, 1)

    # Concatenate: shape (40, C+2)
    dpixel = np.concatenate([sampled_vals, doy_sin, doy_cos], axis=-1)

    # → torch tensor with batch dim: (1, 40, C+2)
    return torch.tensor(dpixel, dtype=torch.float32).unsqueeze(0)

# ─────────────────────────────────────────────
# 4.  NORMALISE BAND VALUES
#     TESSERA uses global statistics. These are the approximate values
#     from the paper/codebase — replace with exact ones from the repo's
#     normalization CSV files (sentinel1.csv / agnostic.csv) if you have them.
# ─────────────────────────────────────────────
# S2: per-band mean and std (surface reflectance, scaled 0–10000)
S2_MEAN = np.array([492.0, 765.0, 866.0, 979.0, 1241.0,
                    1607.0, 1684.0, 1746.0, 1956.0, 1339.0])
S2_STD  = np.array([289.0, 338.0, 384.0, 421.0,  430.0,
                     566.0,  592.0,  596.0,  761.0,  539.0])

# S1: [VV, VH] mean and std (linear power or dB — match your data scale)
S1_MEAN = np.array([-12.5, -19.5])   # dB approximate
S1_STD  = np.array([  4.5,   5.0])

s2_norm = (s2_values - S2_MEAN) / S2_STD
s1_norm = (s1_values - S1_MEAN) / S1_STD

# ─────────────────────────────────────────────
# 5.  RUN INFERENCE
# ─────────────────────────────────────────────
s2_tensor = build_dpixel(s2_norm, s2_doys).to(device)  # (1, 40, 12)
s1_tensor = build_dpixel(s1_norm, s1_doys).to(device)  # (1, 40,  4)

with torch.no_grad():
    # The combined model expects (s1, s2) as separate inputs
    embedding = model(s1_tensor, s2_tensor)  # → (1, 128)

embedding_np = embedding.squeeze(0).cpu().numpy()  # → (128,)
print("Embedding shape:", embedding_np.shape)       # (128,)
print("Embedding sample:", embedding_np[:8])

TypeError: unsupported operand type(s) for -: 'ellipsis' and 'float'